# Earth pilot

Use the project’s `.venv` kernel. Results are saved in `runs/`.

In [ ]:
%matplotlib inline

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Video, display

from gravity_adapt.config import load_config
from gravity_adapt.evaluate import run_evaluation
from gravity_adapt.train import run_training

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
runs_dir = project_root / "runs"

## 1. Parameters

In [ ]:
evaluation_seed = 10000     # Seed for the first evaluation reset.
evaluation_episodes = 10    # Number of evaluation episodes.
record_video = True         # Save a video of each evaluation episode.
smoothing_episodes = 100    # Episodes in the trailing mean on plots.

config = load_config(project_root / "configs" / "earth.yaml")   # Base training configuration.
config.run.name = "earth_pilot"                                 # Run directory label.
config.run.seed = 42                                            # Training random seed.
config.run.device = "cpu"                                       # PyTorch device: cpu, mps, or cuda.
config.run.torch_threads = 1                                    # CPU threads used by PyTorch.
config.run.output_dir = str(runs_dir)                           # Parent directory for training artifacts.
config.environment.num_envs = 4                                 # Environments stepped together.
config.ppo.total_steps = 2_000_000                              # Total training transitions across all environments.

## 2. Train

In [ ]:
training_dir = run_training(config).directory
checkpoint = training_dir / "checkpoint.pt"
print(f"Training directory: {training_dir}")

## 3. Load training logs

In [ ]:
training_config = load_config(training_dir / "config.yaml")
metadata = json.loads((training_dir / "metadata.json").read_text())
metrics = pd.read_csv(training_dir / "metrics.csv")
episodes = pd.read_csv(training_dir / "episodes.csv")

transitions = metrics["env_steps"].diff()
transitions.iloc[0] = metrics["env_steps"].iloc[0]
rollout_size = training_config.ppo.rollout_steps * training_config.environment.num_envs
full_rollouts = transitions == rollout_size
compute_seconds = metrics["rollout_seconds"] + metrics["update_seconds"]
metrics["steps_per_second"] = transitions / compute_seconds

display(pd.Series({
    "status": metadata["status"],
    "device": training_config.run.device,
    "num_envs": training_config.environment.num_envs,
    "torch_threads": training_config.run.torch_threads,
    "transitions_in_this_run": int(transitions.sum()),
    "completed_episodes_in_this_run": len(episodes),
    "compute_steps_per_second": transitions.sum() / compute_seconds.sum(),
    "elapsed_seconds": metrics["elapsed_seconds"].iloc[-1],
}, name="Training run"))

## 4. Learning curves

Points: raw episodes. Line: trailing mean over `smoothing_episodes`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, column, label in zip(axes, ["return", "length"], ["Raw episode return", "Episode length"], strict=True):
    ax.scatter(episodes["env_steps"], episodes[column], s=8, alpha=0.2, color="tab:blue", label="Episodes")
    smoothed = episodes[column].rolling(smoothing_episodes, min_periods=1).mean()
    ax.plot(episodes["env_steps"], smoothed, color="tab:orange", label=f"Trailing mean ({smoothing_episodes} episodes)")
    ax.set(xlabel="Environment transitions", ylabel=label)
    ax.legend()
    ax.grid(alpha=0.2)
plt.show()

## 5. PPO diagnostics

Metrics are averaged per update; gradient norm is measured before clipping.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
columns = ["policy_loss", "critic_loss", "entropy", "approx_kl", "clip_fraction", "grad_norm"]
for ax, column in zip(axes.flat, columns, strict=True):
    ax.plot(metrics.loc[full_rollouts, "env_steps"], metrics.loc[full_rollouts, column])
    if not full_rollouts.all():
        ax.scatter(
            metrics.loc[~full_rollouts, "env_steps"], metrics.loc[~full_rollouts, column],
            color="C1", marker="x", label="Partial rollout", zorder=3,
        )
        ax.legend()
    ax.set(xlabel="Environment transitions", ylabel=column)
    ax.grid(alpha=0.2)
plt.show()

## 6. Runtime

Timing: full rollouts. Throughput: all rollouts, excluding file writing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(metrics["env_steps"], metrics["steps_per_second"])
axes[0].set(xlabel="Environment transitions", ylabel="Transitions / second")
for column in ["rollout_seconds", "update_seconds"]:
    axes[1].plot(metrics.loc[full_rollouts, "env_steps"], metrics.loc[full_rollouts, column], label=column)
axes[1].set(xlabel="Environment transitions", ylabel="Seconds / update", title="Full rollouts")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()

## 7. Independent evaluation

Mean actions and frozen normalization.

In [ ]:
evaluation_dir = run_evaluation(
    checkpoint, seed=evaluation_seed, episodes=evaluation_episodes,
    video=record_video, output_dir=runs_dir,
).directory
print(f"Evaluation directory: {evaluation_dir}")

evaluation_metadata = json.loads((evaluation_dir / "metadata.json").read_text())
evaluation_config = load_config(evaluation_dir / "config.yaml")
evaluation = pd.read_csv(evaluation_dir / "episodes.csv")
display(evaluation_metadata["evaluation"])
print(f"Evaluation seed: {evaluation_config.run.seed}")
print(f"Gravity: {evaluation_config.environment.gravity}")
display(evaluation.set_index("episode"))
display(evaluation[["return", "length", "x_displacement"]].agg(["mean", "std", "min", "median", "max"]))
reached_limit = evaluation["truncated"] & ~evaluation["terminated"]
print(f"Reached time limit without termination: {reached_limit.sum()}/{len(evaluation)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, column, label in zip(
    axes, ["return", "length", "x_displacement"],
    ["Raw return", "Episode length", "Forward displacement (m)"], strict=True,
):
    ax.bar(evaluation["episode"], evaluation[column])
    ax.set(xlabel="Evaluation episode", ylabel=label)
    ax.axhline(0, color="black", linewidth=0.6)
plt.show()

## 8. Video

Video indices start at zero. Clear notebook outputs before committing.

In [ ]:
video_index = 0
video_files = sorted(
    (evaluation_dir / "videos").glob("*.mp4"),
    key=lambda path: int(path.stem.rsplit("-", 1)[-1]),
)
if video_files:
    display(Video(filename=str(video_files[video_index]), embed=True))
else:
    print("No videos in this evaluation. Set record_video=True for a new evaluation.")